Alles gut


The leapfrog method

In [ ]:
#3 body: leapfrog method KDK
import numpy as np
import matplotlib.pyplot as plt

#constants
G = 1  #make it easy for arbitrary body

#position vectors at t = 0 (initial conditions set to arbitrary("simple" to make check easier) values for my code)
r1 = np.array([-0.5, 0.0, 0.0])  #body 1
r2 = np.array([0.5, 0.0, 0.0])   #body 2
r3 = np.array([5.0, 0.0, 0.0])   #body 3
r = np.array([r1, r2, r3])

#set arbitrary masses
m = np.array([1.0, 2.0, 0.2])

#set initial velocities to start the KDK (not too large so that they cant escape)
v1 = np.array([0.0, 0.7, 0.0])
v2 = np.array([0.0, -0.5, 0.0])
v3 = np.array([0.0, 0.0, 1.0])
v = np.array([v1, v2, v3])

#velocity correction so that the centre of mass o fthe system stays at one point -> plot and read
p = m[0]*v[0] + m[1]*v[1] + m[2]*v[2]
vcm = p / (m[0] + m[1] + m[2])    # centre of mass velocity
v = v - vcm

#∆t
dt = 0.001

#outer acceleration function!!!
def accelerations(r, m):
    a = np.zeros_like(r)
    for j in range(len(m)):
        for k in range(j + 1, len(m)):
            d = r[k] - r[j]
            dcube = (np.linalg.norm(d))**3
            a[j] += G * m[k] * (d/dcube)
            a[k] -= G * m[j] * (d/dcube)
    return a

a = accelerations(r,m)

#defining energy for energy conservation and erro rplot   (E = kin-pot)
def energy(r, v, m):
    kin = 0.0
    for j in range(len(m)):     #kinetic energy
        kin += 1/2 * m[j] * (np.linalg.norm(v[j]))**2

    pot = 0.0
    for j in range(len(m)):   #potntial energy 
        for k in range(j + 1, len(m)):
            pot += G * (m[j]*m[k]) / np.linalg.norm(r[k] - r[j])

    ener = kin - pot 
    return ener


#storing data for the energy calculation later
positions = [r.copy()]
velocities = [v.copy()]

#main loop and KDK implementation (half kick -> drift -> half kick)
n_steps = 10000

for i in range(n_steps):
    v += a * dt / 2   #half kick I
    r += v *dt        #drift
    a = accelerations(r,m)    #new a due to new positions(acceleration only depends on position)
    v += a * dt / 2   #half kick II
    positions.append(r.copy())
    velocities.append(v.copy())

#turn positions & velocities to array to use for energy 
positions = np.array(positions)
velocities = np.array(velocities)

#energy loop for comparison and plot
energies = []
for i in range(n_steps + 1):
    En = energy(positions[i], velocities[i], m)
    energies.append(En)
energies = np.array(energies)    

#relative error
E0 = energies[0]
diff = energies - E0
e_rel = diff / E0
e_rel = np.abs(e_rel)



Runge-Kutta method (fourth order)

In [ ]:
G=1
m=np.array([1,1,1])
r=np.array([[-1.0,0.0,0.0],[1.0,0.0,0.0],[0.0,0.0,0.0]])
v=np.array([[0.464445,0.396060,0.0],[0.464445,0.396060,0.0],[-2*0.464445,-2*0.396060,0.0]])
h=0.001

def acceleration(r,m,G):
    a=np.zeros_like(r)
    for i in range (3):
        for j in range (3):
            if(i!=j):
                a[i]+=-G*m[j]*(r[i]-r[j])/(np.linalg.norm(r[i]-r[j]))**3
    return a

def RK4(r,v,m,G,h):
    k1_r=v
    k1_v=acceleration(r,m,G)

    k2_r=v+h/2*k1_v
    k2_v=acceleration(r+h/2*k1_r,m,G)

    k3_r=v+h/2*k2_v
    k3_v=acceleration(r+h/2*k2_r,m,G)

    k4_r=v+h*k3_v
    k4_v=acceleration(r+h*k3_r,m,G)

    r_new=r+h/6*(k1_r+2*k2_r+2*k3_r+k4_r)
    v_new=v+h/6*(k1_v+2*k2_v+2*k3_v+k4_v)

    return r_new,v_new

steps=10000
p=np.zeros((steps,3,3))
s=np.zeros((steps,3,3))

for step in range (steps):
    next_r,next_v=RK4(r,v,m,G,h)
    p[step]=next_r
    s[step]=next_v
    r=next_r
    v=next_v